# Day 63 - the memory hierarchy and the roofline

Every performance number in this series turned out to be a memory number. This notebook
is the model underneath: the hierarchy measured on this machine, the roofline that
decides whether a kernel is limited by arithmetic or by traffic, and tiling - including
FlashAttention - as the only way to change which side you are on.

In [ ]:
import numpy as np
from memory_hierarchy import (hierarchy_table, measure_sequential, measure_random,
                              Machine, roofline_table, batch_crossover, matmul_traffic,
                              tile_sweep, flash_io, flash_scaling, little, strided_cost)

for r in hierarchy_table():
    print(f"{r['name']:26} {r['gbs']:9,.0f} GB/s   {r['ns']:6,.0f} ns")

registers                    100,000 GB/s        1 ns
shared memory / LDS           20,000 GB/s       20 ns
L2 cache                       7,000 GB/s      200 ns
HBM                            5,300 GB/s      600 ns
host memory over PCIe             64 GB/s    2,000 ns
another node over RDMA            50 GB/s   10,000 ns


## 1. Measured here: in order against at random

Same machine, same bytes. The prefetcher hides the hierarchy from one of them.

In [ ]:
for r in measure_sequential():
    print(f"{r['kb']:7d} KB   {r['gbs']:6.1f} GB/s")
print()
for r in measure_random():
    print(f"{r['kb']:7d} KB   {r['ns_per_access']:6.2f} ns per access")

     64 KB     16.5 GB/s
    256 KB     20.6 GB/s
   1024 KB     21.9 GB/s
   4096 KB     22.2 GB/s
  16384 KB     19.4 GB/s
  65536 KB     19.2 GB/s
 262144 KB     18.3 GB/s

     16 KB     1.64 ns per access
     64 KB     1.44 ns per access
    256 KB     1.45 ns per access
   1024 KB     2.77 ns per access
   4096 KB     8.14 ns per access
  16384 KB    10.04 ns per access
  65536 KB    11.93 ns per access


## 2. The roofline

A machine is two numbers; their ratio is the balance. A kernel below it cannot be
compute-bound however it is written.

In [ ]:
rows, m = roofline_table()
print(f"peak {m.flops/1e12:.0f} TFLOP/s, {m.bw/1e9:.0f} GB/s, balance {m.balance:.0f} FLOPs/byte\n")
for r in rows:
    print(f"{r['name']:36} {r['intensity']:10.2f}  {r['bound']:8} {r['ms']:7.2f} ms")

peak 1300 TFLOP/s, 5300 GB/s, balance 245 FLOPs/byte

decode: one linear (GEMV)                  1.00  memory      0.02 ms
prefill: one linear (GEMM)              3185.78  compute     0.37 ms
decode: attention over KV                  1.00  memory      0.01 ms
prefill: attention (no tiling)           128.00  memory      0.41 ms
RMSNorm                                    1.00  memory      0.01 ms
residual add                               0.17  memory      0.02 ms
softmax over the vocabulary                1.25  memory      0.00 ms
decode: the whole model                    1.00  memory     26.42 ms


## 3. Batching moves a kernel along the roofline

In [ ]:
bc, balance = batch_crossover()
for r in bc:
    print(f"batch {r['batch']:5d}  {r['intensity']:9.2f} FLOPs/byte  {r['bound']:8}"
          f"  {r['per_token_us']:7.2f} us per token")

batch     1       1.00 FLOPs/byte  memory      22.16 us per token
batch     4       4.00 FLOPs/byte  memory       5.54 us per token
batch    16      15.96 FLOPs/byte  memory       1.39 us per token
batch    64      63.43 FLOPs/byte  memory       0.35 us per token
batch   256     247.17 FLOPs/byte  compute      0.09 us per token
batch  1024     896.00 FLOPs/byte  compute      0.09 us per token


## 4. Tiling, and FlashAttention as the same argument

In [ ]:
t = matmul_traffic()
print(f"untiled {t['naive_bytes']/1e9:.1f} GB   tiles of {t['tile']} "
      f"{t['tiled_bytes']/1e9:.2f} GB   {t['ratio']:.0f}x less")
for r in tile_sweep():
    print(f"  T={r['tile']:4d} {'fits' if r['fits'] else 'DOES NOT FIT':>12}  {r['gb']:8.2f} GB")

print()
f = flash_io()
print(f"score matrix {f['score_matrix_gb']:.2f} GB resident; standard moves "
      f"{f['standard_gb']:.2f} GB, tiled {f['tiled_gb']:.2f} GB ({f['ratio']:.1f}x)")
print(f"the matrix is {f['memory_ratio']:,.0f}x the shared memory the kernel runs in")
for r in flash_scaling():
    print(f"  seq {r['seq']:6d}  matrix {r['score_matrix_gb']:8.2f} GB  ratio {r['ratio']:.1f}x")

untiled 549.8 GB   tiles of 104 5.35 GB   103x less
  T=   8         fits     68.79 GB
  T=  16         fits     34.43 GB
  T=  32         fits     17.25 GB
  T=  64         fits      8.66 GB
  T= 128 DOES NOT FIT      4.36 GB
  T= 256 DOES NOT FIT      2.21 GB
  T= 512 DOES NOT FIT      1.14 GB

score matrix 1.07 GB resident; standard moves 4.43 GB, tiled 1.81 GB (2.4x)
the matrix is 6,554x the shared memory the kernel runs in
  seq    512  matrix     0.02 GB  ratio 2.0x
  seq   2048  matrix     0.27 GB  ratio 2.4x
  seq   8192  matrix     4.29 GB  ratio 2.5x
  seq  32768  matrix    68.72 GB  ratio 2.5x


## 5. Latency and cache lines

In [ ]:
print(little())
for r in strided_cost():
    print(f"stride {r['stride']:3d}  useful {r['useful_mb']:7.2f} MB  fetched {r['fetched_mb']:7.2f} MB"
          f"  {r['efficiency']:6.1%}  {r['seconds']*1e3:5.2f} ms")

{'bandwidth_gbs': 5300.0, 'latency_ns': 600.0, 'bytes_in_flight': 3180000.0, 'requests_in_flight': 24843.75, 'bytes_per_request': 128}
stride   1  useful   16.78 MB  fetched   16.78 MB  100.0%   0.87 ms
stride   2  useful    8.39 MB  fetched   16.78 MB   50.0%   1.16 ms
stride   4  useful    4.19 MB  fetched   16.78 MB   25.0%   0.45 ms
stride   8  useful    2.10 MB  fetched   16.78 MB   12.5%   0.55 ms
stride  16  useful    1.05 MB  fetched   16.78 MB    6.2%   0.63 ms
stride  32  useful    0.52 MB  fetched    8.39 MB    6.2%   0.40 ms


## 6. Tests

In [ ]:
m = Machine()
rows, _ = roofline_table()
by = {r['name']: r for r in rows}
assert by['decode: one linear (GEMV)']['bound'] == 'memory'
assert by['prefill: one linear (GEMM)']['bound'] == 'compute'
bc, _ = batch_crossover()
assert bc[0]['bound'] == 'memory' and bc[-1]['bound'] == 'compute'
t = matmul_traffic(n=1024, tile=64)
assert abs(t['ratio'] - 64) < 3
assert flash_io(seq=16384)['memory_ratio'] > 1000
print('all assertions passed')

all assertions passed
